# AWGN

> What the receiver gets from the transmitted symbols: y = h √P x + n.

In [ ]:
#| default_exp channel.awgn

In [ ]:
#| hide
from nbdev.showdoc import *

The channel has two halves. The first is the coefficient `h` of a transmission, from a `Fading` (see
`channel.fading`): the path loss, the fading and the phase. The second, here, applies it to the transmitted symbols
`x`:

`y = h √P x + n`, with complex Gaussian noise `n ~ CN(0, noise_power)` (AWGN).

`h` and `P` broadcast against the symbols. With one value per packet, i.e. shape `(..., 1)`, the channel is flat
and constant during the packet. This is Sionna's `ApplyFlatFadingChannel` followed by its `AWGN`, on numpy arrays
or torch tensors. `ApplyChannel` is the same as a stage of a link chain (see `radio.link.chain`).

In [ ]:
#| export
from __future__ import annotations

import numpy as np

from comm_core.block import Block, _is_torch

In [ ]:
#| export
def awgn(
    like, # Array or tensor whose shape (and device) the noise takes
    noise_power, # Noise power (variance of the complex noise)
    rng=None, # numpy: a Generator or seed; torch: a torch.Generator or None
):
    "Complex Gaussian noise, CN(0, noise_power)."
    std = (noise_power / 2) ** 0.5
    if _is_torch(like):
        import torch
        re = torch.randn(like.shape, generator=rng, device=like.device)
        im = torch.randn(like.shape, generator=rng, device=like.device)
        return torch.complex(re, im) * std
    rng = np.random.default_rng(rng)
    shape = np.shape(like)
    return (rng.standard_normal(shape) + 1j * rng.standard_normal(shape)) * std


def apply_channel(
    symbols, # Transmitted symbols x, (..., num_symbols)
    h, # Channel coefficients (complex), broadcast against the symbols, e.g. (..., 1): one per packet
    power, # Transmit power P, broadcast against the symbols
    noise_power, # Noise power at the receiver
    rng=None, # Random generator of the noise (see `awgn`)
):
    "What the receiver gets: y = h √P x + n."
    signal = h * power ** 0.5 * symbols
    return signal + awgn(signal, noise_power, rng)

## As a stage

`ApplyChannel` gives `y` from the symbols `x`, the channel of each symbol `h_sym` (the waveform's, `√P` included,
see `waveform`) and each transmission's noise power `no`: the receiver's noise plus the interference, treated as
Gaussian noise (see `channel.interference`). The noise comes from the stage's random generator (the chain's).

In [ ]:
#| export
def _per_row(no, like):
    "Each transmission's value, broadcast against its symbols (a tensor of `like`'s real type on its device, or numpy)."
    if _is_torch(like):
        import torch
        return torch.as_tensor(np.asarray(no, float), dtype=like.real.dtype, device=like.device).reshape(
            (len(no),) + (1,) * (like.dim() - 1))
    return np.asarray(no, float).reshape((len(no),) + (1,) * (np.ndim(like) - 1))


class ApplyChannel(Block):
    "What the receiver gets: y = h x + n, n ~ CN(0, no) with each transmission's noise power (the interference included)."

    outputs = ('y',)

    def call(
        self,
        x, # The symbols, (B, ..., S)
        h_sym, # The channel of each symbol, √P included, broadcast against x
        no, # Each transmission's noise power, (B,)
    ):
        "y, like x."
        if _is_torch(x):
            import torch
            gen = torch.Generator(device=x.device).manual_seed(int(self.rng.integers(2 ** 62)))
            real = x.real.dtype
            noise = torch.complex(torch.randn(x.shape, generator=gen, device=x.device, dtype=real),
                                  torch.randn(x.shape, generator=gen, device=x.device, dtype=real))
            return h_sym * x + noise * torch.sqrt(_per_row(no, x) / 2)
        return np.stack([apply_channel(x[b], h_sym[b], 1.0, float(no[b]), self.rng) for b in range(len(x))])

### Tests

In [ ]:
from comm_core.modulation.constellation import BPSK
from comm_core.radio.system.phy_abstraction import bpsk_bit_error_rate

The simulated bit error rate of BPSK is the analytic one (see `radio.system.phy_abstraction`):

In [ ]:
# the simulated bit error rate of BPSK is the analytic one (see `radio.system.phy_abstraction`)
bpsk = BPSK()
rng = np.random.default_rng(0)
b = rng.integers(0, 2, 200_000)
for snr_db in (0, 4, 7):
    noise_power = 10 ** (-snr_db / 10)                       # P = 1, |h| = 1
    h = np.exp(1j * 0.7)
    errors = (bpsk.demodulate(apply_channel(bpsk.modulate(b), h, 1.0, noise_power, rng), h) != b).mean()
    expected = bpsk_bit_error_rate(10 ** (snr_db / 10))
    assert abs(errors - expected) < 0.1 * expected + 2e-4, (snr_db, errors, expected)
n = awgn(np.zeros(100_000), 2.0, rng)
assert n.dtype == complex and abs((np.abs(n) ** 2).mean() - 2.0) < 0.05
# as a stage: each row its own noise power, from the stage's generator
stage = ApplyChannel()
stage.reset(seed=0)
y = stage(np.ones((2, 50_000)), np.full((2, 1), 2.0), np.array([0.1, 1.0]))
assert np.allclose(y.mean(1), 2.0, atol=0.02) and np.allclose((np.abs(y - 2.0) ** 2).mean(1), [0.1, 1.0], rtol=0.03)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()